In [1]:
## Linear Programming - Finding the Optimal Multiplication
import pulp

n = 233
max_P = [2,1,2,3,6,9,18,30,56,99]
mul_u = [1, 3, 5, 8, 11, 15, 18, 22,27,30]
mul_F = [1, 3, 6, 9, 13, 15, 22, 24,30,33]
mu_d_u = [[mul_F[i]*mul_u[j] for j in range(10)] for i in range(10)]

prob = pulp.LpProblem("Minimize_Additions", pulp.LpMinimize)

# x[i][j]: The number of choices u_i=j+1 in the i-th group
x = [[pulp.LpVariable(f"x_{i+1}_{j+1}", lowBound=0, upBound=max_P[i], cat='Integer') for j in range(10)] for i in range(10)]
# y[j]: y takes j+1
y = [pulp.LpVariable(f"y_{j+1}", cat='Binary') for j in range(10)]

# The sum of x[i][j] in each group does not exceed max_P[i]
for i in range(10):
    prob += pulp.lpSum([x[i][j] for j in range(10)]) <= max_P[i]

# y one-hot
prob += pulp.lpSum(y) == 1

# Constraint: sum_{i=1}^8 sum_{j=1}^8 x[i][j] * (i+1) * (j+1) + sum_{j=1}^8 (j+1)*y[j] == 2*n-1
prob += (
    pulp.lpSum([x[i][j] * (i+1) * (j+1) for i in range(10) for j in range(10)])
    + pulp.lpSum([(j+1)*y[j] for j in range(10)])
    == 2*n-1
)
# Objective function
prob += (
    pulp.lpSum([x[i][j] * mul_F[i]*mul_u[j] for i in range(10) for j in range(10)])
    + pulp.lpSum([mul_u[j]*y[j] for j in range(10)])
)

prob.solve()

print("Status:", pulp.LpStatus[prob.status])
u=[]
num=[]
for i in range(10):
    for j in range(10):
        val = pulp.value(x[i][j])
        if val > 0.5:
            u.append(j+1)
            num.append(int(val))
            print(f"x_{i+1}_{j+1} = {val}")
            print(f"u_{i+1} = {j+1}")  
for j in range(10):
    if pulp.value(y[j]) > 0.5:
        u.append(j+1)
        print(f"y = {j+1}")
print("Objective function value =", pulp.value(prob.objective))
print(u)
print(num)

Welcome to the CBC MILP Solver 
Version: 2.10.3 
Build Date: Dec 15 2019 

command line - /home/kekai/miniconda3/lib/python3.12/site-packages/pulp/apis/../solverdir/cbc/linux/i64/cbc /tmp/f5cab577ab0d43c3a81e6df49aeb1ee4-pulp.mps -timeMode elapsed -branch -printingOptions all -solution /tmp/f5cab577ab0d43c3a81e6df49aeb1ee4-pulp.sol (default strategy 1)
At line 2 NAME          MODEL
At line 3 ROWS
At line 17 COLUMNS
At line 568 RHS
At line 581 BOUNDS
At line 692 ENDATA
Problem MODEL has 12 rows, 110 columns and 220 elements
Coin0008I MODEL read with 0 errors
Option for timeMode changed from cpu to elapsed
Continuous objective value is 1340.57 - 0.00 seconds
Cgl0004I processed model has 12 rows, 110 columns (110 integer (20 of which binary)) and 220 elements
Cutoff increment increased from 1e-05 to 0.9999
Cbc0012I Integer solution of 1342 found by DiveCoefficient after 0 iterations and 0 nodes (0.01 seconds)
Cbc0012I Integer solution of 1341 found by DiveCoefficient after 4 iterations an